# Notebook 07 — Seaborn Pairplot of PCA Components by Regime

This notebook is the **only** place seaborn's pairplot is imported and used.
It is deliberately kept separate from the main pipeline because:
- `sns.pairplot` is slow (30–90 seconds for 5 components × 300 rows)
- It should never block a headless pipeline run
- Keeping seaborn imports isolated here makes the rest of the package lighter

**Prerequisites:** Run the pipeline through at least step 3.
```
python run_pipeline.py --steps 1,2,3
```

In [1]:
import sys
from pathlib import Path

# Allow notebook to find the package without `pip install -e .`
repo_root = Path().resolve().parent
sys.path.insert(0, str(repo_root / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

from market_regime import DATA_DIR, OUTPUT_DIR
from market_regime.config import load
from market_regime import plotting

cfg = load()
print("Data dir:", DATA_DIR)

Data dir: /Users/glestryc/personal/github_repos/claude-scratch-work/data


In [2]:
# ── Load PCA components and cluster labels ─────────────────────────────────
pca_path = DATA_DIR / "regimes" / "pca_components.parquet"
labels_path = DATA_DIR / "regimes" / "cluster_labels.parquet"

pca_df = pd.read_parquet(pca_path)
labels_df = pd.read_parquet(labels_path)
labels = labels_df["balanced_cluster"]

print(f"PCA components: {pca_df.shape}")
print(f"Labels: {labels.value_counts().to_dict()}")

PCA components: (304, 5)
Labels: {2: 62, 3: 62, 0: 62, 4: 60, 1: 58}


In [3]:
# ── Load regime names (auto-suggested or manually overridden) ─────────────
regime_names: dict[int, str] = {}
for name_path in [
    Path(str(DATA_DIR).replace("data", "config")) / "regime_labels.yaml",  # overrides
    DATA_DIR / "regimes" / "regime_names_suggested.yaml",
]:
    if name_path.exists():
        with open(name_path) as f:
            raw = yaml.safe_load(f) or {}
        names = {int(k): v for k, v in raw.items() if not str(k).startswith("#")}
        if names:
            regime_names = names
            break

print("Regime names:", regime_names)

Regime names: {0: 'Falling CPI / GDP Contracting / Rates Falling', 1: 'High Inflation / Rising CPI / Strong Growth', 2: 'Rising CPI / Strong Growth / Strong Real Growth', 3: 'Low Inflation / Falling CPI / Strong Real Growth', 4: 'Falling CPI / Weak/Neg Growth / Weak Real Growth'}


In [4]:
# ── Build pairplot DataFrame ───────────────────────────────────────────────
N_COMPONENTS = 5   # adjust if you want more/fewer panels

pc_cols = [c for c in pca_df.columns if c.startswith("PC")][:N_COMPONENTS]
df_plot = pca_df[pc_cols].copy()
df_plot["Regime"] = labels.reindex(df_plot.index).map(
    lambda x: regime_names.get(int(x), f"R{int(x)}") if pd.notna(x) else "?"
)

# Color palette aligned to regime IDs
unique_ids = sorted(labels.dropna().astype(int).unique())
palette = {
    regime_names.get(i, f"R{i}"): plotting.CUSTOM_COLORS[i % len(plotting.CUSTOM_COLORS)]
    for i in unique_ids
}

print(f"Pairplot on {len(df_plot)} quarters, {len(pc_cols)} PCA components")
print(f"Color palette: {palette}")

Pairplot on 304 quarters, 5 PCA components
Color palette: {'Falling CPI / GDP Contracting / Rates Falling': '#0000d0', 'High Inflation / Rising CPI / Strong Growth': '#d00000', 'Rising CPI / Strong Growth / Strong Real Growth': '#f48c06', 'Low Inflation / Falling CPI / Strong Real Growth': '#8338ec', 'Falling CPI / Weak/Neg Growth / Weak Real Growth': '#50a000'}


In [5]:
# ── Draw pairplot (slow — ~30–90 seconds) ─────────────────────────────────
g = sns.pairplot(
    df_plot,
    hue="Regime",
    palette=palette,
    diag_kind="kde",
    plot_kws={"alpha": 0.55, "s": 18, "edgecolors": "none"},
    diag_kws={"fill": True, "alpha": 0.45},
)
g.figure.suptitle("PCA Pairplot by Regime", y=1.02, fontsize=14)

# Optionally save
out_path = OUTPUT_DIR / "plots" / "07_pca_pairplot.png"
out_path.parent.mkdir(parents=True, exist_ok=True)
g.figure.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved to {out_path}")
plt.show()

Saved to /Users/glestryc/personal/github_repos/claude-scratch-work/outputs/plots/07_pca_pairplot.png


/Users/glestryc/tmp/ipykernel_44578/3383486196.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# ── Optional: also try market_code as hue if available ────────────────────
if "market_code" in labels_df.columns:
    mc = labels_df["market_code"]
    df_mc = pca_df[pc_cols].copy()
    df_mc["market_code"] = mc.reindex(df_mc.index).map(
        lambda x: f"Grok-{int(x)}" if pd.notna(x) else "?"
    )
    g2 = sns.pairplot(
        df_mc,
        hue="market_code",
        diag_kind="kde",
        plot_kws={"alpha": 0.55, "s": 18, "edgecolors": "none"},
    )
    g2.figure.suptitle("PCA Pairplot by market_code (Grok labels)", y=1.02, fontsize=14)
    out2 = OUTPUT_DIR / "plots" / "07_pca_pairplot_market_code.png"
    g2.figure.savefig(out2, dpi=150, bbox_inches="tight")
    print(f"Saved to {out2}")
    plt.show()
else:
    print("No market_code column in cluster_labels — skipping grok overlay.")

Saved to /Users/glestryc/personal/github_repos/claude-scratch-work/outputs/plots/07_pca_pairplot_market_code.png


/Users/glestryc/tmp/ipykernel_44578/2545575982.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
